# Upload a Dataset to a Pinecone Vector DB

In [36]:
from pinecone import Pinecone, ServerlessSpec, AwsRegion
from pinecone_datasets import load_dataset, list_datasets
import os

### Show available datasets

In [2]:
for ds in list_datasets():
    print(ds)

ANN_DEEP1B_d96_angular
ANN_Fashion-MNIST_d784_euclidean
ANN_GIST_d960_euclidean
ANN_GloVe_d100_angular
ANN_GloVe_d200_angular
ANN_GloVe_d25_angular
ANN_GloVe_d50_angular
ANN_LastFM_d64_angular
ANN_MNIST_d784_euclidean
ANN_NYTimes_d256_angular
ANN_SIFT1M_d128_euclidean
amazon_toys_quora_all-MiniLM-L6-bm25
cohere-768
it-threat-data-test
it-threat-data-train
langchain-python-docs-text-embedding-ada-002
mnist
movielens-user-ratings
msmarco-v1-bm25-allMiniLML6V2
nq-768-tasb
quora_all-MiniLM-L6-bm25-100K
quora_all-MiniLM-L6-bm25
quora_all-MiniLM-L6-v2_Splade-100K
quora_all-MiniLM-L6-v2_Splade
squad-text-embedding-ada-002
wikipedia-simple-text-embedding-ada-002-100K
wikipedia-simple-text-embedding-ada-002-50K
wikipedia-simple-text-embedding-ada-002
yfcc-100K-filter-euclidean
yfcc-10M-filter-euclidean
yfcc-10M-filter-euclidean
youtube-transcripts-text-embedding-ada-002


### Load and Preprocess the Data

In [77]:
ds = load_dataset("amazon_toys_quora_all-MiniLM-L6-bm25")

ds.head()

Loading documents parquet files: 100%|██████████| 1/1 [00:02<00:00,  2.11s/it]


,id,values,sparse_values,metadata,blob
0,eac7efa5dbd3d667f26eb3d3ab504464,"[0.0077547780238091946, -0.02774387039244175, ...","{'indices': [2182291806, 4287202515, 148124445...",{'amazon_category_and_sub_category': 'Hobbies ...,{'text': 'Hornby 2014 Catalogue (Hornby): Pr...
1,b17540ef7e86e461d37f3ae58b7b72ac,"[0.002257382730022073, -0.03035414218902588, 0...","{'indices': [2118423442, 2177509083, 224097760...",{'amazon_category_and_sub_category': 'Hobbies ...,{'text': 'FunkyBuys® Large Christmas Holiday E...
2,348f344247b0c1a935b1223072ef9d8a,"[-0.003095218911767006, 0.016020774841308594, ...","{'indices': [2349888478, 3814962844, 310417642...",{'amazon_category_and_sub_category': 'Hobbies ...,{'text': 'CLASSIC TOY TRAIN SET TRACK CARRIAGE...
3,e12b92dbb8eaee78b22965d2a9bbbd9f,"[-0.024034591391682625, -0.048526741564273834,...","{'indices': [2182291806, 719182917, 1942275469...",{'amazon_category_and_sub_category': 'Hobbies ...,{'text': 'HORNBY Coach R4410A BR Hawksworth Co...
4,e33a9adeed5f36840ccc227db4682a36,"[-0.07078640908002853, 0.009733847342431545, 0...","{'indices': [2182291806, 2415375917, 369727517...",{'amazon_category_and_sub_category': 'Hobbies ...,{'text': 'Hornby 00 Gauge 0-4-0 Gildenlow Salt...


In [78]:
ds_no_blob = ds.documents.drop('blob', axis=1)
ds_no_blob = ds_no_blob.drop('metadata', axis=1)
ds_no_blob['metadata'] = ds.documents['blob']
ds_no_blob

,id,values,sparse_values,metadata
0,eac7efa5dbd3d667f26eb3d3ab504464,"[0.0077547780238091946, -0.02774387039244175, ...","{'indices': [2182291806, 4287202515, 148124445...",{'text': 'Hornby 2014 Catalogue (Hornby): Pr...
1,b17540ef7e86e461d37f3ae58b7b72ac,"[0.002257382730022073, -0.03035414218902588, 0...","{'indices': [2118423442, 2177509083, 224097760...",{'text': 'FunkyBuys® Large Christmas Holiday E...
2,348f344247b0c1a935b1223072ef9d8a,"[-0.003095218911767006, 0.016020774841308594, ...","{'indices': [2349888478, 3814962844, 310417642...",{'text': 'CLASSIC TOY TRAIN SET TRACK CARRIAGE...
3,e12b92dbb8eaee78b22965d2a9bbbd9f,"[-0.024034591391682625, -0.048526741564273834,...","{'indices': [2182291806, 719182917, 1942275469...",{'text': 'HORNBY Coach R4410A BR Hawksworth Co...
4,e33a9adeed5f36840ccc227db4682a36,"[-0.07078640908002853, 0.009733847342431545, 0...","{'indices': [2182291806, 2415375917, 369727517...",{'text': 'Hornby 00 Gauge 0-4-0 Gildenlow Salt...
...,...,...,...,...
9995,44d6967f083825a5de36ad4865a65bcd,"[-0.028194401413202286, 0.020815584808588028, ...","{'indices': [537607261, 3331209479, 595801246,...",{'text': 'Batman 1966 TV Series Action Figures...
9996,08f0747b6fc6687215ffb994c3a6fb32,"[0.0005118539556860924, 0.052324745804071426, ...","{'indices': [2844778355, 1822615852, 429458463...","{'text': 'Star Wars Costume, Kids Stormtrooper..."
9997,bf6cc073f8f24e6e338190fa16f6ee9d,"[-0.0863356739282608, 0.08705731481313705, 0.0...","{'indices': [3576897476, 3982843675, 359430131...",{'text': 'Defiance Lawkeeper Metal Badge Prop ...
9998,cd783d0b8b44e631b9788b203eaaefae,"[-0.0582226887345314, -0.049953095614910126, -...","{'indices': [1790916742, 653372694, 3654969415...",{'text': 'Justice League of America Series 3 G...


In [84]:
for i in range(len(ds_no_blob)):
    indices = ds_no_blob['sparse_values'][i]['indices']
    values = ds_no_blob['sparse_values'][i]['values']
    text = ds_no_blob['metadata'][i]['text']
    ds_no_blob['sparse_values'][i]['indices'] = indices[:2048]
    ds_no_blob['sparse_values'][i]['values'] = values[:2048]
    ds_no_blob['metadata'][i]['text'] = text[:40000]

In [85]:
maxlen1 = 0
maxlen2 = 0
maxlen3 = 0
for i in range(len(ds_no_blob['sparse_values'])):
    if len(ds_no_blob['sparse_values'][i]['indices']) > maxlen1:
        maxlen1 = len(ds_no_blob['sparse_values'][i]['indices'])
    
    if len(ds_no_blob['sparse_values'][i]['values']) > maxlen2:
        maxlen2 = len(ds_no_blob['sparse_values'][i]['values'])

    if len(ds_no_blob['metadata'][i]['text']) > maxlen3:
        maxlen3 = len(ds_no_blob['metadata'][i]['text'])

print(maxlen1, maxlen2, maxlen3)

2048 2048 40000


In [86]:
ds_no_blob

,id,values,sparse_values,metadata
0,eac7efa5dbd3d667f26eb3d3ab504464,"[0.0077547780238091946, -0.02774387039244175, ...","{'indices': [2182291806, 4287202515, 148124445...",{'text': 'Hornby 2014 Catalogue (Hornby): Pr...
1,b17540ef7e86e461d37f3ae58b7b72ac,"[0.002257382730022073, -0.03035414218902588, 0...","{'indices': [2118423442, 2177509083, 224097760...",{'text': 'FunkyBuys® Large Christmas Holiday E...
2,348f344247b0c1a935b1223072ef9d8a,"[-0.003095218911767006, 0.016020774841308594, ...","{'indices': [2349888478, 3814962844, 310417642...",{'text': 'CLASSIC TOY TRAIN SET TRACK CARRIAGE...
3,e12b92dbb8eaee78b22965d2a9bbbd9f,"[-0.024034591391682625, -0.048526741564273834,...","{'indices': [2182291806, 719182917, 1942275469...",{'text': 'HORNBY Coach R4410A BR Hawksworth Co...
4,e33a9adeed5f36840ccc227db4682a36,"[-0.07078640908002853, 0.009733847342431545, 0...","{'indices': [2182291806, 2415375917, 369727517...",{'text': 'Hornby 00 Gauge 0-4-0 Gildenlow Salt...
...,...,...,...,...
9995,44d6967f083825a5de36ad4865a65bcd,"[-0.028194401413202286, 0.020815584808588028, ...","{'indices': [537607261, 3331209479, 595801246,...",{'text': 'Batman 1966 TV Series Action Figures...
9996,08f0747b6fc6687215ffb994c3a6fb32,"[0.0005118539556860924, 0.052324745804071426, ...","{'indices': [2844778355, 1822615852, 429458463...","{'text': 'Star Wars Costume, Kids Stormtrooper..."
9997,bf6cc073f8f24e6e338190fa16f6ee9d,"[-0.0863356739282608, 0.08705731481313705, 0.0...","{'indices': [3576897476, 3982843675, 359430131...",{'text': 'Defiance Lawkeeper Metal Badge Prop ...
9998,cd783d0b8b44e631b9788b203eaaefae,"[-0.0582226887345314, -0.049953095614910126, -...","{'indices': [1790916742, 653372694, 3654969415...",{'text': 'Justice League of America Series 3 G...


### Upsert to Pinecone

In [92]:
pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index_name = "toys-index"
ns = 'namespace1'

if not pc.has_index(name=index_name):
    index_config = pc.create_index(
        name=index_name,
        dimension=ds.metadata.dense_model.dimension,
        spec=ServerlessSpec(cloud="aws", region=AwsRegion.US_EAST_1)
    )

In [ ]:
index = pc.Index(index_name)

index.upsert_from_dataframe(ds_no_blob, namespace=ns)


sending upsert requests:  20%|██        | 2000/10000 [06:38<26:33,  5.02it/s] 




















sending upsert requests: 100%|██████████| 10000/10000 [00:28<00:00, 350.98it/s]


UpsertResponse(upserted_count=10000, _response_info={'raw_headers': {'date': 'Mon, 09 Mar 2026 19:17:19 GMT', 'content-type': 'application/json', 'content-length': '21', 'connection': 'keep-alive', 'x-pinecone-request-lsn': '36', 'x-pinecone-request-logical-size': '1828324', 'x-pinecone-request-latency-ms': '767', 'x-envoy-upstream-service-time': '208', 'x-pinecone-response-duration-ms': '769', 'grpc-status': '0', 'server': 'envoy'}})

### Test with a Query

In [106]:
from sentence_transformers import SentenceTransformer

sentences = ["This is an example sentence", "Each sentence is converted"]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode(sentences)
print(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13485.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[[ 6.76568747e-02  6.34958595e-02  4.87131178e-02  7.93049559e-02
   3.74480262e-02  2.65277666e-03  3.93748954e-02 -7.09844334e-03
   5.93614392e-02  3.15370075e-02  6.00980520e-02 -5.29051870e-02
   4.06067856e-02 -2.59308219e-02  2.98427679e-02  1.12689380e-03
   7.35149458e-02 -5.03819436e-02 -1.22386619e-01  2.37028636e-02
   2.97265276e-02  4.24768589e-02  2.56338008e-02  1.99519680e-03
  -5.69190979e-02 -2.71598604e-02 -3.29035223e-02  6.60248771e-02
   1.19007044e-01 -4.58791591e-02 -7.26214722e-02 -3.25839967e-02
   5.23413755e-02  4.50552665e-02  8.25299136e-03  3.67023535e-02
  -1.39414975e-02  6.53918982e-02 -2.64272541e-02  2.06395140e-04
  -1.36643685e-02 -3.62809785e-02 -1.95043236e-02 -2.89738607e-02
   3.94270457e-02 -8.84091035e-02  2.62426934e-03  1.36713842e-02
   4.83062975e-02 -3.11565548e-02 -1.17329188e-01 -5.11690192e-02
  -8.85287449e-02 -2.18962003e-02  1.42986486e-02  4.44167629e-02
  -1.34814689e-02  7.43392408e-02  2.66382825e-02 -1.98762361e-02
   1.79191

In [119]:
from pinecone import SearchQuery
import numpy as np

# Define the query
query = "Hornby Trains"
query_vector = model.encode(query)
query_vector = np.squeeze(query_vector).tolist()

results = index.query(
    namespace=ns,
    vector=query_vector,
    top_k=20,
    include_metadata=True
)

In [122]:
for match in results.matches:
    print(f"Score : {match['score']} \nProduct: {match['metadata']['text']}")
    print()

Score : 0.711777687 
Product: Hornby Pennine Express Train Set (Hornby): 
 LOVELY TRADITIONAL LAYOUT 
 Technical Details Manufacturer recommended age:5 years and up    Additional Information ASINB00ANYSJWO Best Sellers Rank 335,242 in Toys & Games (See top 100) #568 in Toys & Games > Model Trains & Railway Sets > Rail Vehicles > Trains Delivery Destinations:Visit the Delivery Destinations Help page to see where this item can be delivered. Date First Available14 Dec. 2012    Feedback  Would you like to update product info or give feedback on images?

Score : 0.711401 
Product: Hornby The Western Spirit OO Gauge Electric Train Set (Hornby): 
 Product Description There were times when trains like the Western Spirit hauled not only freight but individual coaches They ran on small branch lines that ran from the main lines to the small rural towns and villages of the British IslesThis set includes a smart 4 wheeled locomotive whose heritage can be traced back to the end of the late 19th cent